# Multi-Agent Systems
### LangGraph · CrewAI · AutoGen — All powered by Groq

---
**Task used across all 3 frameworks:** *"Suggest 3 trending AI research topics and write a one-paragraph summary for each."*

Same task. Three different coordination styles. Watch how each framework handles it differently.

| Framework | Coordination Style | our Lecture Concept |
|-----------|-------------------|----------------------|
| **LangGraph** | Graph nodes + Supervisor routing | Hierarchical / Supervisor Architecture |
| **CrewAI** | Role-based crew (Researcher → Analyst → Writer) | Sequential Handoffs + Specialization |
| **AutoGen** | Conversational agents (back-and-forth) | Network / Peer-to-Peer Communication |


##  Step 1 — Enter Your Groq API Key

In [1]:
import getpass
import os

GROQ_API_KEY = getpass.getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("API Key set successfully!")

Enter your Groq API Key: ··········
API Key set successfully!


## Step 2 — Install Dependencies if required
> Run this once. Takes ~1-2 minutes.

In [2]:
!pip install -q langgraph langchain-groq crewai pyautogen groq litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.

---
# FRAMEWORK 1 — LangGraph
## Supervisor Architecture with Graph Routing

```
        ┌─────────────┐
        │  SUPERVISOR │   ← decides who works next
        └──────┬──────┘
               │
      ┌────────┴────────┐
      ▼                 ▼
 [Researcher]       [Writer]
  (finds topics)  (writes summaries)
```

**Key concepts from our lecture:**
- Nodes = Agents
- Edges = Routing logic
- Shared State flows through the graph
- Supervisor decides which agent runs next → `FINISH` when done

In [3]:
# ── LangGraph: Supervisor + Worker Agents ──────────────────────────────────
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated

# Shared State — flows through every node in the graph
class AgentState(TypedDict):
    task: str
    topics: str        # filled by Researcher
    summaries: str     # filled by Writer
    next: str          # filled by Supervisor → controls routing

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

# ── NODE 1: Researcher ─────────────────────────────────────────────────────
def researcher_node(state: AgentState) -> AgentState:
    print("[Researcher Agent] Finding trending AI topics...")
    response = llm.invoke(
        f"You are a research specialist. List exactly 3 trending AI research topics "
        f"as a numbered list. Be concise. Task: {state['task']}"
    )
    return {**state, "topics": response.content, "next": "writer"}

# ── NODE 2: Writer ─────────────────────────────────────────────────────────
def writer_node(state: AgentState) -> AgentState:
    print(" [Writer Agent] Writing summaries...")
    response = llm.invoke(
        f"You are a technical writer. For each of these AI topics, write a "
        f"2-sentence summary suitable for students.\n\nTopics:\n{state['topics']}"
    )
    return {**state, "summaries": response.content, "next": "FINISH"}

# ── NODE 3: Supervisor ─────────────────────────────────────────────────────
def supervisor_node(state: AgentState) -> AgentState:
    # Routing logic: if no topics yet → go to researcher, else → writer
    if not state.get("topics"):
        print("[Supervisor] Routing to → Researcher")
        return {**state, "next": "researcher"}
    elif not state.get("summaries"):
        print("[Supervisor] Routing to → Writer")
        return {**state, "next": "writer"}
    else:
        print(" [Supervisor] All done → FINISH")
        return {**state, "next": "FINISH"}

# ── Build the Graph ────────────────────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)

# Entry point → always start at supervisor
graph.set_entry_point("supervisor")

# Conditional routing based on state["next"]
graph.add_conditional_edges(
    "supervisor",
    lambda s: s["next"],
    {"researcher": "researcher", "writer": "writer", "FINISH": END}
)

# After each worker → return to supervisor for next decision
graph.add_edge("researcher", "supervisor")
graph.add_edge("writer", "supervisor")

app = graph.compile()

# ── Run it ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("LANGGRAPH — SUPERVISOR ARCHITECTURE")
print("=" * 60)

result = app.invoke({
    "task": "Find 3 trending AI research topics and summarize each",
    "topics": "",
    "summaries": "",
    "next": ""
})

print("\n TOPICS FOUND:")
print(result["topics"])
print("\n SUMMARIES WRITTEN:")
print(result["summaries"])

LANGGRAPH — SUPERVISOR ARCHITECTURE
[Supervisor] Routing to → Researcher
[Researcher Agent] Finding trending AI topics...
[Supervisor] Routing to → Writer
 [Writer Agent] Writing summaries...
 [Supervisor] All done → FINISH

 TOPICS FOUND:
1. **Explainable AI (XAI)**: Developing techniques to make AI decisions transparent and interpretable.
2. **Transfer Learning**: Enabling AI models to apply knowledge learned from one task to other related tasks.
3. **Adversarial Robustness**: Improving AI models' resistance to adversarial attacks and data perturbations.

 SUMMARIES WRITTEN:
Here are the 2-sentence summaries for each topic:

**Explainable AI (XAI)**: Explainable AI (XAI) is a subfield of artificial intelligence that focuses on developing techniques to make AI decisions transparent and interpretable, allowing users to understand the reasoning behind the predictions or recommendations made by AI models. By providing insights into AI decision-making processes, XAI aims to increase trust

---
#  FRAMEWORK 2 — CrewAI
## Role-Based Crew with Sequential Handoffs

```
  [Researcher] ──► [Analyst] ──► [Writer]
   finds topics    validates    final report
```

**Key concepts from our lecture:**
- Each agent has a `role`, `goal`, and `backstory`
- Tasks are passed **sequentially** (handoff pattern)
- `Process.sequential` = ordered pipeline
- Clean responsibility boundaries → modularity

In [4]:
!pip install "crewai[litellm]" --upgrade -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 899.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you 

In [5]:
# ── CrewAI: Role-Based Crew ────────────────────────────────────────────────
from crewai import Agent, Task, Crew, Process
import os
import getpass

GROQ_API_KEY = getpass.getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ── Use LiteLLM model string format ───────────────────────────────────────
groq_model = "groq/llama-3.1-8b-instant"   # ← key fix: prefix with "groq/"

# ── Define Agents ─────────────────────────────────────────────────────────
researcher = Agent(
    role="AI Research Specialist",
    goal="Identify the 3 most trending topics in AI research right now",
    backstory="You are an expert in tracking cutting-edge AI research papers and trends.",
    llm=groq_model,   # ← pass string, not ChatGroq object
    verbose=True
)

analyst = Agent(
    role="Research Analyst",
    goal="Validate and add context to the AI topics provided",
    backstory="You critically evaluate research topics for accuracy and relevance to 2024-2025.",
    llm=groq_model,
    verbose=True
)

writer = Agent(
    role="Technical Writer",
    goal="Write clear, student-friendly summaries of AI research topics",
    backstory="You specialize in making complex AI concepts accessible to university students.",
    llm=groq_model,
    verbose=True
)

# ── Define Tasks ───────────────────────────────────────────────────────────
task_research = Task(
    description="List exactly 3 trending AI research topics as a numbered list with one-line descriptions.",
    expected_output="A numbered list of 3 AI research topics with brief descriptions.",
    agent=researcher
)

task_analyze = Task(
    description="Review the 3 topics from the researcher. Confirm they are relevant and add one key insight per topic.",
    expected_output="The same 3 topics with one validated insight added to each.",
    agent=analyst
)

task_write = Task(
    description="Write a 2-sentence student-friendly summary for each of the 3 validated AI topics.",
    expected_output="A final report with 3 topics, each with a 2-sentence summary.",
    agent=writer
)

# ── Assemble & Run the Crew ────────────────────────────────────────────────
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_research, task_analyze, task_write],
    process=Process.sequential,
    verbose=True
)

print("=" * 60)
print("CREWAI — ROLE-BASED SEQUENTIAL CREW")
print("=" * 60)

result = crew.kickoff()
print("\nFINAL CREW OUTPUT:")
print(result)

Enter your Groq API Key: ··········
CREWAI — ROLE-BASED SEQUENTIAL CREW


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d9d258af-386f-4562-8d22-104a41e40fd6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: List exactly 3 trending AI research topics as a numbered list with one-line descriptions.                │
│  ID: 6c578a91-f613-4127-8ecc-1cda1cd4435e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Research Specialist                                                                                  │
│                                                                                                                 │
│  Task: List exactly 3 trending AI research topics as a numbered list with one-line descriptions.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Final Report: 3 AI Topics with Student-Friendly Summaries**                                                  │
│                                                                                                                 │
│  1. **Explainable AI (XAI)**                                                                                    │
│                                                                                                                 │
│  Explainable AI (XAI) is a field of research that focuses on making AI models more transparent and              │
│  understandable. By developing techniques such as model-agnostic explanations and feature importance, XAI       │
│  enables practitioners to gain insight into the decision-making process of complex AI models, making it easier  │
│  to trust and understand their outputs.                                                                         │
│                                                                                                                 │
│  This area of research has shown promise in generating explanations for AI predictions and decisions,           │
│  regardless of the model's architecture or complexity. Techniques such as SHAP values and partial dependence    │
│  plots provide crucial information about how AI models arrive at their conclusions, facilitating more informed  │
│  decision-making.                                                                                               │
│                                                                                                                 │
│  2. **Large Language Models (LLMs)**                                                                            │
│                                                                                                                 │
│  Large Language Models (LLMs) are artificial intelligence systems designed to generate human-like language and  │
│  tackle complex tasks such as conversational dialogue, text summarization, and text-to-image synthesis.         │
│  Researchers are working to enhance the capabilities of LLMs by integrating them with other modalities, such    │
│  as images, to create multimodal models that can process and generate text and images simultaneously.           │
│                                                                                                                 │
│  The potential applications of multimodal LLMs are vast, and researchers are exploring their use in tasks such  │
│  as image captioning, visual question answering, and text-to-image synthesis. By integrating text and images,   │
│  LLMs can provide more comprehensive and effective language models that can better understand and respond to    │
│  complex queries.                                                                                               │
│                                                                                                                 │
│  3. **Causality in AI**                                                                                         │
│                                                                                                                 │
│  Causality in AI is a research area that focuses on developing AI systems that can accurately identify and      │
│  understand the underlying causes of events. This enabl

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a 2-sentence student-friendly summary for each of the 3 validated AI topics.                       │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d9d258af-386f-4562-8d22-104a41e40fd6                                                                       │
│  Final Output: **Final Report: 3 AI Topics with Student-Friendly Summaries**                                    │
│                                                                                                                 │
│  1. **Explainable AI (XAI)**                                                                                    │
│                                                                                                                 │
│  Explainable AI (XAI) is a field of research that focuses on making AI models more transparent and              │
│  understandable. By developing techniques such as model-agnostic explanations and feature importance, XAI       │
│  enables practitioners to gain insight into the decision-making process of complex AI models, making it easier  │
│  to trust and understand their outputs.                                                                         │
│                                                                                                                 │
│  This area of research has shown promise in generating explanations for AI predictions and decisions,           │
│  regardless of the model's architecture or complexity. Techniques such as SHAP values and partial dependence    │
│  plots provide crucial information about how AI models arrive at their conclusions, facilitating more informed  │
│  decision-making.                                                                                               │
│                                                                                                                 │
│  2. **Large Language Models (LLMs)**                                                                            │
│                                                                                                                 │
│  Large Language Models (LLMs) are artificial intelligence systems designed to generate human-like language and  │
│  tackle complex tasks such as conversational dialogue, text summarization, and text-to-image synthesis.         │
│  Researchers are working to enhance the capabilities of LLMs by integrating them with other modalities, such    │
│  as images, to create multimodal models that can process and generate text and images simultaneously.           │
│                                                                                                                 │
│  The potential applications of multimodal LLMs are vast, and researchers are exploring their use in tasks such  │
│  as image captioning, visual question answering, and text-to-image synthesis. By integrating text and images,   │
│  LLMs can provide more comprehensive and effective language models that can better understand and respond to    │
│  complex queries.                                                                                               │
│                                                                                                                 │
│  3. **Causality in AI**                                                                                         │
│                                                                                                                 │
│  Causality in AI is a research area that focuses on developing AI systems that can accurately identify and      │
│  understand the underlying causes of events. This enab


FINAL CREW OUTPUT:
**Final Report: 3 AI Topics with Student-Friendly Summaries**

1. **Explainable AI (XAI)**

Explainable AI (XAI) is a field of research that focuses on making AI models more transparent and understandable. By developing techniques such as model-agnostic explanations and feature importance, XAI enables practitioners to gain insight into the decision-making process of complex AI models, making it easier to trust and understand their outputs.

This area of research has shown promise in generating explanations for AI predictions and decisions, regardless of the model's architecture or complexity. Techniques such as SHAP values and partial dependence plots provide crucial information about how AI models arrive at their conclusions, facilitating more informed decision-making.

2. **Large Language Models (LLMs)**

Large Language Models (LLMs) are artificial intelligence systems designed to generate human-like language and tackle complex tasks such as conversational dialogu



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [6]:
#  Hierarchical CrewAI (Manager -> Researcher; Manager finalizes)
import os, time
from getpass import getpass
from crewai import Agent, Task, Crew, Process, LLM

# --- API key (Groq) ---
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter API key for Groq: ")

GROQ_MODEL = "groq/llama-3.1-8b-instant"

def groq_llm(temp=0.0):
    # Very small completions to reduce TPM usage
    return LLM(
        model=GROQ_MODEL,
        api_key=os.environ["GROQ_API_KEY"],
        temperature=temp,
        top_p=0.8,
        max_tokens=300,   # tiny budgets keep calls cheap + under rate limits
        timeout=20,
    )

# --- Coworker (kept to ONE to keep it minimal) ---
researcher = Agent(
    role="Researcher",
    goal="Return 2 sourced bullet points on Pakistan’s K–12 EdTech (2024–2025).",
    backstory="Very concise market researcher.",
    llm=groq_llm(0.1),
    memory=False,
    verbose=False,
    max_rpm=20,
)

# --- Manager ---
# Key: we FORCE the manager to use the tool with EXACT schema: coworker, task, context (ALL strings).
manager = Agent(
    role="Manager",
    goal=(
        "Coordinate coworkers and deliver a final 60–80 word summary. "
        "First, DELEGATE to Researcher to fetch 2 sourced bullets. "
        "Then, WRITE the final summary yourself (do NOT delegate writing)."
    ),
    backstory=(
        "IMPORTANT TOOL RULES:\n"
        "- When using 'Delegate work to coworker', ALWAYS pass a flat JSON object with exactly these string keys:\n"
        '  {"coworker":"Researcher","task":"<plain string>","context":"<plain string>"}\n'
        "- Never pass nested dicts. Never omit 'coworker'."
    ),
    llm=groq_llm(0.0),
    memory=False,
    verbose=False,
    max_rpm=15,
)

# --- One high-level task (Manager will orchestrate) ---
main_task = Task(
    description=(
        "Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). "
        "Process: 1) Delegate to Researcher: 'Return exactly 2 bullets with source tags'. "
        "2) Then YOU (Manager) write the final summary using ONLY those bullets. No citations."
    ),
    expected_output="One paragraph, 60–80 words.",

)

# --- Crew (hierarchical) ---
crew = Crew(
    agents=[researcher],            # Manager is NOT listed here
    tasks=[main_task],
    process=Process.hierarchical,   # Manager-led
    manager_agent=manager,          # Explicit manager
    memory=False,
    verbose=True,
    full_output=False,              # less metadata
)


def run_with_retry(fn, retries=2, base_delay=4):
    for i in range(retries + 1):
        try:
            return fn()
        except Exception as e:
            if i == retries:
                raise
            time.sleep(base_delay * (2 ** i))  # 4s, 8s

result = run_with_retry(lambda: crew.kickoff(inputs={"topic": "Pakistani K–12 EdTech (2024–2025)"}))

print("\n=== FINAL OUTPUT ===\n")
print(getattr(result, "raw", None) or result or "No final output.")


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 90eaa737-9cc4-4310-b81b-2958b27c0668                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: method callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  """
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: method callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  """


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). Process: 1)          │
│  Delegate to Researcher: 'Return exactly 2 bullets with source tags'. 2) Then YOU (Manager) write the final     │
│  summary using ONLY those bullets. No citations.                                                                │
│  ID: a9640f48-a422-4ced-a0e9-bf68a2f29341                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Manager                                                                                                 │
│                                                                                                                 │
│  Task: Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). Process: 1)          │
│  Delegate to Researcher: 'Return exactly 2 bullets with source tags'. 2) Then YOU (Manager) write the final     │
│  summary using ONLY those bullets. No citations.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). Process: 1)          │
│  Delegate to Researcher: 'Return exactly 2 bullets with source tags'. 2) Then YOU (Manager) write the final     │
│  summary using ONLY those bullets. No citations.                                                                │
│  Agent: Manager                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')


=== FINAL OUTPUT ===

The tool result does not meet the requirement of providing a 60-80 word summary. To meet the requirement, I will write the final summary using the provided bullets.

Pakistan's K-12 EdTech market is expected to grow significantly from 2024 to 2025, driven by increasing demand for online learning platforms and digital content. By 2025, most Pakistani schools are expected to adopt EdTech solutions, focusing on AI-powered adaptive learning, virtual classrooms, and mobile learning apps. This shift towards digital education will likely improve learning outcomes and increase access to quality education for students across the country.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 90eaa737-9cc4-4310-b81b-2958b27c0668                                                                       │
│  Final Output: The tool result does not meet the requirement of providing a 60-80 word summary. To meet the     │
│  requirement, I will write the final summary using the provided bullets.                                        │
│                                                                                                                 │
│  Pakistan's K-12 EdTech market is expected to grow significantly from 2024 to 2025, driven by increasing        │
│  demand for online learning platforms and digital content. By 2025, most Pakistani schools are expected to      │
│  adopt EdTech solutions, focusing on AI-powered adaptive learning, virtual classrooms, and mobile learning      │
│  apps. This shift towards digital education will likely improve learning outcomes and increase access to        │
│  quality education for students across the country.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# FRAMEWORK 3 — AutoGen
## Conversational Peer-to-Peer Agents

```
  [UserProxy] ◄──► [ResearchBot] ◄──► [CriticBot]
   initiates       proposes topics     challenges + refines
```

**Key concepts from our lecture:**
- Agents **talk to each other** like a group chat
- No fixed pipeline — they negotiate and refine
- Network / Peer-to-Peer collaboration style
- `GroupChat` = shared message bus (Blackboard Model concept!)

In [7]:
!pip install "autogen-agentchat" "autogen-ext[openai]" -q



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [8]:
import asyncio
import os
from getpass import getpass
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

# Groq via OpenAI-compatible client
model_client = OpenAIChatCompletionClient(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "family": "unknown",
    }
)

# ── Agents ────────────────────────────────────────────────────────────────
research_bot = AssistantAgent(
    name="ResearchBot",
    system_message=(
        "You are an AI research expert. Propose 3 trending AI research topics "
        "with a short explanation each. Be concise."
    ),
    model_client=model_client,
)

critic_bot = AssistantAgent(
    name="CriticBot",
    system_message=(
        "You are a critical reviewer. Pick the BEST 3 topics from ResearchBot, "
        "briefly explain why they matter, and write a 2-sentence student-friendly "
        "summary for each. Then say TERMINATE."
    ),
    model_client=model_client,
)

# ── Team with termination condition ───────────────────────────────────────
termination = TextMentionTermination("TERMINATE")

team = RoundRobinGroupChat(
    participants=[research_bot, critic_bot],
    termination_condition=termination,
    max_turns=6,
)

print("=" * 60)
print("AUTOGEN v0.4 — CONVERSATIONAL PEER-TO-PEER AGENTS")
print("=" * 60)

# ── Run ───────────────────────────────────────────────────────────────────
await Console(team.run_stream(
    task="Identify 3 trending AI research topics and provide student-friendly summaries."
))


/usr/local/lib/python3.12/dist-packages/autogen_ext/models/openai/_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


AUTOGEN v0.4 — CONVERSATIONAL PEER-TO-PEER AGENTS
---------- TextMessage (user) ----------
Identify 3 trending AI research topics and provide student-friendly summaries.
---------- TextMessage (ResearchBot) ----------
Here are 3 trending AI research topics:

1. **Explainable AI (XAI)**: 
Imagine you're using a medical diagnosis AI, but you want to know why it made a specific decision. XAI helps explain the reasoning behind AI's decisions, making them more transparent and trustworthy.

2. **Causal AI**:
Causal AI focuses on understanding the causes and effects of events. It can help in areas like predicting disease outbreaks, analyzing the impact of climate change, or identifying the factors contributing to a company's success.

3. **Multimodal Learning & Transformers**: 
Multimodal learning enables AI to process various types of data (like images, speech, text, and sensor data) simultaneously. Transformer-based architectures can then analyze and learn new patterns from these multiple d

TaskResult(messages=[TextMessage(id='af26e6f0-54a8-4687-86ad-041ac08e2e40', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 17, 13, 11, 19, 619397, tzinfo=datetime.timezone.utc), content='Identify 3 trending AI research topics and provide student-friendly summaries.', type='TextMessage'), TextMessage(id='9385b86b-da9a-4af8-b8dd-87483ffd29c6', source='ResearchBot', models_usage=RequestUsage(prompt_tokens=73, completion_tokens=180), metadata={}, created_at=datetime.datetime(2026, 5, 17, 13, 11, 20, 850517, tzinfo=datetime.timezone.utc), content="Here are 3 trending AI research topics:\n\n1. **Explainable AI (XAI)**: \nImagine you're using a medical diagnosis AI, but you want to know why it made a specific decision. XAI helps explain the reasoning behind AI's decisions, making them more transparent and trustworthy.\n\n2. **Causal AI**:\nCausal AI focuses on understanding the causes and effects of events. It can help in areas like predicting disease out

---
# 📊 Comparison Summary

| | LangGraph | CrewAI | AutoGen |
|---|---|---|---|
| **Style** | Graph + Supervisor routing | Sequential role-based crew | Conversational back-and-forth |
| **our lecture pattern** | Supervisor / Hierarchical | Handoff / Sequential | Network / Peer-to-Peer |
| **Control** | You define the graph explicitly | Framework manages the pipeline | Agents negotiate dynamically |
| **Best for** | Complex branching workflows | Structured team workflows | Brainstorming, debate, refinement |
| **Debugging** |  Easy (graph trace) |  Easy (step logs) |  Harder (emergent conversation) |
| **Flexibility** | Very High | Moderate | High |

> **Takeaway for students:** Same task, three different coordination strategies. The task didn't change — the *agent architecture* did. This is the core idea behind Multi-Agent System design.